# 🚀 DataMate: Runner & Benchmark on Vast.ai (RTX 5090 / 4090 / A100)
This notebook provides a seamless, 1-click execution of the **DataMate** agent pipeline using **vLLM** on high-performance GPU instances rented from **Vast.ai** (e.g., **Dual RTX 5090 32GB**, **RTX 4090**, or **A100**).


## 1. Hardware & GPU Acceleration Verification
Verify that your instance has recognized the GPUs and CUDA runtime is active.


In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)} - {torch.cuda.get_device_properties(i).total_memory / (1024**3):.1f} GB VRAM")


## 2. Check Repository & Switch to Branch `loc`
Ensure we are in the `datamate` repository and have pulled the latest Phase 10 / Phase 11 code.


In [ ]:
import os
REPO_URL = "https://github.com/vohoangloc-32/datamate.git"
BRANCH = "loc"

if not os.path.exists("src/predict.py"):
    print("Cloning repository...")
    !git clone --depth 1 -b {BRANCH} {REPO_URL} datamate
    os.chdir("datamate")
else:
    print("Already inside datamate repository. Updating code...")
    !git fetch origin && git checkout {BRANCH} && git reset --hard origin/{BRANCH}

!git log -1 --oneline


## 3. Install Dependencies & Run Verification Tests
Install vLLM and DataMate requirements, then execute the full unit test suite (196 tests).


In [ ]:
!pip install -q vllm openpyxl xlrd tabulate pytest huggingface_hub pandas
!pytest tests/ -q


## 4. Download Model Weights to Local NVMe SSD
Downloads `Qwen/Qwen2.5-Coder-14B-Instruct-GPTQ-Int8` via fast parallel threads.


In [ ]:
from huggingface_hub import snapshot_download
import os

MODEL_REPO = "Qwen/Qwen2.5-Coder-14B-Instruct-GPTQ-Int8"
LOCAL_MODEL_DIR = os.path.expanduser("~/model_cache/Qwen2.5-Coder-14B-Instruct-GPTQ-Int8")

os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)
print(f"Downloading {MODEL_REPO} to {LOCAL_MODEL_DIR}...")
snapshot_download(
    repo_id=MODEL_REPO,
    local_dir=LOCAL_MODEL_DIR,
    local_dir_use_symlinks=False,
    max_workers=8
)
print("✅ Model weights downloaded successfully!")


## 5. Run DataMate Prediction Pipeline
Execute the pipeline on your chosen dataset.
- Mode `train_unseen_500`: The official 500-question unseen validation benchmark.
- Mode `public_test`: The competition public test set.
- On Dual RTX 5090, 500 questions will finish in approximately 10-15 minutes!


In [ ]:
import os
import time

# Multi-GPU / Single-GPU settings
# If using 2x RTX 5090, VLLM_TP can be 1 (GPU 0) or 2 (tensor parallel across both GPUs)
os.environ["VLLM_TP"] = "1"
os.environ["VLLM_DTYPE"] = "auto"
os.environ["VLLM_MAX_MODEL_LEN"] = "16384"
os.environ["VLLM_MAX_NEW_TOKENS"] = "1024"

# Execution mode: 'train_unseen_500' | 'train_500' | 'public_test' | 'train_200'
MODE = "train_unseen_500"
CONFIDENCE_THRESHOLD = "0.45"

WORKDIR = os.path.abspath(".")
if MODE == "public_test":
    DATA_DIR = os.path.join(WORKDIR, "data/public_test")
    QUESTIONS_FILE = os.path.join(WORKDIR, "data/public_test/questions.jsonl")
elif MODE == "train_unseen_500":
    DATA_DIR = os.path.join(WORKDIR, "data/train")
    QUESTIONS_FILE = os.path.join(WORKDIR, "data/sample_questions_unseen_500.jsonl")
elif MODE == "train_500":
    DATA_DIR = os.path.join(WORKDIR, "data/train")
    QUESTIONS_FILE = os.path.join(WORKDIR, "data/sample_questions_500.jsonl")
elif MODE == "train_200":
    DATA_DIR = os.path.join(WORKDIR, "data/train")
    QUESTIONS_FILE = os.path.join(WORKDIR, "data/sample_questions_200.jsonl")
else:
    raise ValueError(f"Unknown MODE: {MODE}")

OUTPUT_DIR = os.path.join(WORKDIR, "results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 60)
print(f"🚀 Starting DataMate Pipeline on Vast.ai...")
print(f"Execution Mode   : {MODE}")
print(f"Data Directory   : {DATA_DIR}")
print(f"Questions File   : {QUESTIONS_FILE}")
print(f"Output Directory : {OUTPUT_DIR}")
print(f"Model Directory  : {LOCAL_MODEL_DIR}")
print(f"Confidence Thresh: {CONFIDENCE_THRESHOLD}")
print("=" * 60)

t_start = time.time()
!python src/predict.py \
    --data-dir {DATA_DIR} \
    --questions-file {QUESTIONS_FILE} \
    --output-dir {OUTPUT_DIR} \
    --model-dir {LOCAL_MODEL_DIR} \
    --confidence-threshold {CONFIDENCE_THRESHOLD} \
    --log-level INFO

elapsed = time.time() - t_start
print(f"✅ Pipeline finished in {elapsed:.1f} seconds ({elapsed/60:.2f} minutes)!")


## 6. Compute Official Penalized Accuracy (PA) & Benchmark Results
Evaluates predictions against ground truth answers using the competition metric:
$$\text{PA} = \frac{\text{Correct} - \text{Wrong}}{\text{Total}}$$


In [ ]:
import os
import json
import pandas as pd
from src.evaluate import compute_pa

SUBMISSION_PATH = os.path.join(OUTPUT_DIR, "submission.jsonl")
ANSWERS_PATH = os.path.join(DATA_DIR, "answers.jsonl")

if os.path.exists(SUBMISSION_PATH):
    submission_records = [json.loads(l) for l in open(SUBMISSION_PATH, encoding="utf-8-sig") if l.strip()]
    print(f"✅ Generated submission file: {SUBMISSION_PATH} ({len(submission_records)} records)")
    
    if os.path.exists(ANSWERS_PATH):
        print(f"✅ Ground truth answers found: {ANSWERS_PATH}")
        metrics = compute_pa(SUBMISSION_PATH, ANSWERS_PATH)
        print("\n" + "=" * 60)
        print("🏆 DATAMATE OFFICIAL BENCHMARK SCORE (EVALUATED SUBSET)")
        print("=" * 60)
        print(f"Total Questions  : {metrics.total_questions}")
        print(f"Correct (+1 pt)  : {metrics.correct_count} ({metrics.accuracy*100:.1f}%)")
        print(f"Wrong   (-1 pt)  : {metrics.wrong_count} ({metrics.wrong_rate*100:.1f}%)")
        print(f"Abstained (0 pt) : {metrics.abstain_count} ({metrics.abstain_rate*100:.1f}%)")
        print("-" * 60)
        print(f"🔥 Penalized Accuracy (PA): {metrics.penalized_accuracy:+.4f} ({metrics.penalized_accuracy*100:+.2f}%)")
        print("=" * 60)
        
        if metrics.breakdown_by_type:
            print("\n📊 DETAILED BREAKDOWN BY QUESTION TYPE:")
            df_breakdown = pd.DataFrame.from_dict(metrics.breakdown_by_type, orient="index")
            df_breakdown.index.name = "Type"
            df_breakdown = df_breakdown.reset_index()
            df_breakdown["PA"] = df_breakdown["pa"].apply(lambda x: f"{x:+.4f}")
            df_breakdown["Acc%"] = (df_breakdown["correct"] / df_breakdown["total"] * 100).apply(lambda x: f"{x:.1f}%")
            print(df_breakdown[["Type", "total", "correct", "wrong", "abstain", "PA", "Acc%"]].to_string(index=False))
            print("=" * 60)


## 7. Diagnostic Inspector: Error Analysis Breakdown
Generates `error_analysis.csv` and `abstain_analysis.csv` for forensic debugging.


In [ ]:
import os
import json
import pandas as pd
from src.evaluate import score_answer

if os.path.exists(ANSWERS_PATH):
    questions_map = {}
    if os.path.exists(QUESTIONS_FILE):
        with open(QUESTIONS_FILE, "r", encoding="utf-8-sig") as f:
            for line in f:
                if line.strip():
                    q = json.loads(line)
                    questions_map[q["id"]] = q
                    
    gt_map = {}
    with open(ANSWERS_PATH, "r", encoding="utf-8-sig") as f:
        for line in f:
            if line.strip():
                ans = json.loads(line)
                gt_map[ans["id"]] = ans

    sub_map = {}
    with open(SUBMISSION_PATH, "r", encoding="utf-8-sig") as f:
        for line in f:
            if line.strip():
                sub = json.loads(line)
                sub_map[sub["id"]] = sub

    errors = []
    abstains = []
    for q_id, gt in gt_map.items():
        if q_id not in sub_map:
            continue
        sub = sub_map[q_id]
        score = score_answer(sub, gt)
        q_meta = questions_map.get(q_id, {})
        
        row = {
            "id": q_id,
            "business_id": q_meta.get("business_id", ""),
            "question": q_meta.get("question", ""),
            "pred_type": sub.get("type"),
            "gt_type": gt.get("type"),
            "pred_val": sub.get("value"),
            "gt_val": gt.get("value"),
            "as_of_date": q_meta.get("as_of_date", "")
        }
        if score < 0:
            errors.append(row)
        elif score == 0:
            abstains.append(row)
            
    df_err = pd.DataFrame(errors)
    df_abs = pd.DataFrame(abstains)
    
    err_csv = os.path.join(OUTPUT_DIR, "error_analysis.csv")
    abs_csv = os.path.join(OUTPUT_DIR, "abstain_analysis.csv")
    df_err.to_csv(err_csv, index=False, encoding="utf-8-sig")
    df_abs.to_csv(abs_csv, index=False, encoding="utf-8-sig")
    print(f"📊 Saved error analysis report   : {err_csv} ({len(df_err)} errors)")
    print(f"📊 Saved abstain analysis report : {abs_csv} ({len(df_abs)} abstains)")
